# Indices de lexicalité

In [ ]:
#import duckdb
#con = duckdb.connect("../data/ocr-fulltext.db");
#con.sql("SELECT id, decode(content) FROM fulltext_ocr LIMIT 100;").df()

In [ ]:
pip install wordfreq psycopg2-binary

In [ ]:
import re
import unicodedata
from wordfreq import zipf_frequency
#import psycopg2
#import sqlite3
from multiprocessing import Pool, cpu_count
import csv
import duckdb

In [ ]:
# Extrait uniquement les "mots alphabétiques"
def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", text)

In [ ]:
def normalize_text(text):
    # Minuscule
    text = text.lower()

    # Normalisation unicode (accents)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    # Supprime caractères non alphabétiques sauf espaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # Nettoyage espaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# Calcule lexicalité et stats à partir d’un texte
from collections import Counter

def analyze_text(text, threshold=2.5, lang="en", top_n=20):
    normalized = normalize_text(text)
    tokens = tokenize(normalized)
    total_words = len(tokens)

    valid_words = []
    invalid_words = []

    for word in tokens:
        freq = zipf_frequency(word, lang)
        if freq >= threshold:
            valid_words.append(freq)
        else:
            invalid_words.append(word)

    valid_count = len(valid_words)
    invalid_count = len(invalid_words)
    lexicality = (valid_count / total_words * 100) if total_words else 0
    avg_zipf = sum(valid_words) / len(valid_words) if valid_words else 0

    # Top invalides sérialisés en string "mot:count|mot:count|..."
    top_invalid = Counter(invalid_words).most_common(top_n)
    top_invalid_str = "|".join(f"{w}:{c}" for w, c in top_invalid)

    return {
        "total_words": total_words,
        "valid_words": valid_count,
        "invalid_words": invalid_count,
        "lexicality": lexicality,
        "avg_zipf": avg_zipf,
        "top_invalid": top_invalid_str,
    }

In [ ]:
# Lit la base par blocs 
def stream_data(batch_size=1000):
    con = duckdb.connect("../data/ocr-fulltext.db")
    offset = 0
    while True:
        batch = con.sql(f"""
            SELECT id, decode(content) AS content_text
            FROM fulltext_ocr
            LIMIT {batch_size} OFFSET {offset}
        """).fetchall()
        if not batch:
            break
        yield batch
        offset += batch_size

In [ ]:
# Décode le blob et analyse un document
def process_row(row, threshold=2.5, lang="en"):
    doc_id, content = row

    if content is None:
        return None

    # BLOB → bytes
    if isinstance(content, bytes):
        try:
            text = content.decode("utf-8", errors="ignore")
        except:
            # fallback si encodage exotique
            text = content.decode("latin-1", errors="ignore")
    else:
        text = str(content)

    analysis = analyze_text(text, threshold, lang)

    return {
        "id": doc_id,
        **analysis
    }

In [ ]:
# Applique multiprocessing sur les batches de données
def process_database_parallel(batch_size=1000, threshold=2.5, lang="en"):
    n_workers = max(cpu_count() - 1, 1)
    print(f"Workers utilisés: {n_workers}")

    results = []

    with Pool(n_workers) as pool:
        for batch in stream_data(batch_size):

            processed = pool.starmap(
                process_row,
                [(row, threshold, lang) for row in batch]
            )

            results.extend([r for r in processed if r is not None])

    return results

In [ ]:
# Agrège les résultats globaux
def compute_global_summary(report):
    total_docs = len(report)
    total_words = sum(r["total_words"] for r in report)
    total_valid = sum(r["valid_words"] for r in report)
    lexicality = (total_valid / total_words * 100) if total_words else 0

    # Agrège tous les invalides
    global_invalid = Counter()
    for r in report:
        if r.get("top_invalid"):
            for item in r["top_invalid"].split("|"):
                w, c = item.split(":")
                global_invalid[w] += int(c)

    return {
        "total_docs": total_docs,
        "total_words": total_words,
        "lexicality": lexicality,
        "top_invalid_global": global_invalid.most_common(50),
    }

In [ ]:
# Sauvegarde 
def export_report_to_csv(report, output_path="report.csv"):
    if not report:
        return

    fieldnames = report[0].keys()

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(report)

In [ ]:
# Lance tout le pipeline
BATCH_SIZE = 1000
THRESHOLD = 2.5
LANG = "en"

report = process_database_parallel(
    batch_size=BATCH_SIZE,
    threshold=THRESHOLD,
    lang=LANG
)

summary = compute_global_summary(report)

print("\n=== GLOBAL SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

export_report_to_csv(report, "ocr_lexicality-report.csv")

In [ ]:
# Optionnel - exploration des résultats
#df = pd.read_csv("ocr_lexicality-report.csv")
#df["top_invalid_parsed"] = df["top_invalid"].apply(parse_top_invalid)

# Exemple : quels docs ont le plus de mots invalides ?
#df.sort_values("invalid_words", ascending=False).head(20)

In [ ]:
import pandas as pd
df = pd.read_csv("ocr_lexicality-report.csv")

# Distribution par tranches
bins = [0, 50, 70, 85, 95, 100]
labels = ["<50%", "50-70%", "70-85%", "85-95%", ">95%"]
df["quality_tier"] = pd.cut(df["lexicality"], bins=bins, labels=labels)

print(df["quality_tier"].value_counts().sort_index())
print(df.groupby("quality_tier", observed=True)["total_words"].mean())

La corrélation longueur/qualité est nette — les docs courts sont systématiquement moins bons :

<50% : moyenne 337 mots → probablement des pages de garde, index, tableaux
50-70% : moyenne 621 mots → pages partiellement scannées ou très techniques
85-95% : moyenne 2153 mots → vos vrais articles complets, c'est là que la masse est

In [ ]:
df["is_short"] = df["total_words"] < 500

print(df.groupby(["quality_tier", "is_short"], observed=True).size())

In [ ]:
# Les vrais ratés OCR : longs mais mauvaise lexicalité
priority = df[(df["total_words"] >= 500) & (df["lexicality"] < 50)]
print(f"{len(priority)} docs à retraiter en priorité")
priority[["id", "total_words", "lexicality", "avg_zipf"]].to_csv("priority_reprocess.csv", index=False)

# Les cas limites longs : potentiellement récupérables
borderline = df[(df["total_words"] >= 500) & (df["lexicality"].between(50, 70))]
print(f"{len(borderline)} docs borderline")

Les docs bordelines ne sont pas en anglais ou contiennent des listes de substances, des index...

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import warnings

warnings.filterwarnings("ignore")

# ── Style ───────────────────────────────────────────────────────────────
BG      = "#0d1117"
SURFACE = "#161b22"
BORDER  = "#30363d"
TEXT    = "#e6edf3"
MUTED   = "#8b949e"

ACCENT  = "#58a6ff"
GREEN   = "#3fb950"
YELLOW  = "#d29922"
RED     = "#f85149"
ORANGE  = "#db6d28"

TIER_COLORS = {
    "<50%": RED,
    "50-70%": ORANGE,
    "70-85%": YELLOW,
    "85-95%": ACCENT,
    ">95%": GREEN,
}

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": BORDER,
    "axes.labelcolor": TEXT,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "text.color": TEXT,
    "grid.color": BORDER,
    "font.family": "monospace",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
def load_csv(path):
    df = pd.read_csv(path)

    # debug utile en notebook
    print("Colonnes disponibles :", list(df.columns))

    required = ["lexicality", "total_words", "avg_zipf", "invalid_words"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"Colonnes manquantes: {missing}\n"
            f"Colonnes présentes: {list(df.columns)}"
        )

    # features dérivées
    bins   = [0, 50, 70, 85, 95, 100]
    labels = ["<50%", "50-70%", "70-85%", "85-95%", ">95%"]

    df["quality_tier"] = pd.cut(df["lexicality"], bins=bins, labels=labels)
    df["is_short"] = df["total_words"] < 500
    df["log_words"] = np.log10(df["total_words"].clip(lower=1))

    return df

In [ ]:
def fig_overview(df):
    fig = plt.figure(figsize=(16, 9), facecolor=BG)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

    tiers = ["<50%", "50-70%", "70-85%", "85-95%", ">95%"]
    counts = df["quality_tier"].value_counts().reindex(tiers).fillna(0)
    colors = [TIER_COLORS[t] for t in tiers]

    # Distribution tiers
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.bar(tiers, counts.values, color=colors)
    ax1.set_title("Distribution qualité")

    # Histogram lexicalité
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(df["lexicality"], bins=50, color=ACCENT, alpha=0.8)
    ax2.set_title("Lexicalité")

    # Pie
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.pie(counts.values, labels=tiers, colors=colors, autopct="%1.1f%%")

    # Scatter
    ax4 = fig.add_subplot(gs[1, :2])
    sample = df.sample(min(15000, len(df)), random_state=42)
    sc = ax4.scatter(
        sample["log_words"],
        sample["lexicality"],
        c=sample["lexicality"],
        cmap="RdYlGn",
        s=5,
        alpha=0.3
    )
    ax4.set_title("Lexicalité vs taille")

    # Stats
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.axis("off")
    ax5.text(0, 0.9, f"Docs: {len(df):,}")
    ax5.text(0, 0.8, f"Moy: {df['lexicality'].mean():.2f}")

    plt.tight_layout()
    plt.show()

In [ ]:
def fig_length_quality(df):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG)

    tiers = ["<50%", "50-70%", "70-85%", "85-95%", ">95%"]
    colors = [TIER_COLORS[t] for t in tiers]

    # Boxplot
    ax = axes[0]
    data = [df[df["quality_tier"] == t]["total_words"] for t in tiers]
    ax.boxplot(data)
    ax.set_title("Longueur par tier")

    # Court vs long
    ax = axes[1]
    short = [df[(df["quality_tier"]==t) & (df["is_short"])].shape[0] for t in tiers]
    long  = [df[(df["quality_tier"]==t) & (~df["is_short"])].shape[0] for t in tiers]

    x = np.arange(len(tiers))
    ax.bar(x, short, label="court")
    ax.bar(x, long, bottom=short, label="long")
    ax.set_xticks(x)
    ax.set_xticklabels(tiers)
    ax.legend()

    # Heatmap
    ax = axes[2]
    tmp = df.copy()
    tmp["w"] = pd.qcut(tmp["total_words"], 10, labels=False, duplicates="drop")
    tmp["l"] = pd.qcut(tmp["lexicality"], 10, labels=False, duplicates="drop")

    heat = tmp.groupby(["w", "l"]).size().unstack(fill_value=0)
    ax.imshow(heat.values, aspect="auto", cmap="viridis")

    plt.tight_layout()
    plt.show()

In [ ]:
def fig_zipf(df):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG)

    tiers = ["<50%", "50-70%", "70-85%", "85-95%", ">95%"]

    # Zipf
    ax = axes[0]
    data = [df[df["quality_tier"]==t]["avg_zipf"] for t in tiers]
    ax.boxplot(data)
    ax.set_title("Zipf score")

    # scatter
    ax = axes[1]
    sample = df.sample(min(12000, len(df)))
    ax.scatter(sample["avg_zipf"], sample["lexicality"], alpha=0.2, s=5)

    # invalid rate
    ax = axes[2]
    rate = df["invalid_words"] / df["total_words"] * 100
    ax.hist(rate, bins=50)

    plt.tight_layout()
    plt.show()

In [ ]:
df = load_csv("ocr_lexicality-report.csv")

fig_overview(df)
fig_length_quality(df)
fig_zipf(df)